<a href="https://colab.research.google.com/github/vsmalladi/FAIRBio/blob/main/fairbio/notebooks/ga4gh_trs_workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explore Workflows from TRS Endpoints

In [1]:
!pip install git+https://github.com/vsmalladi/FAIRBio

  Cloning https://github.com/vsmalladi/FAIRBio to /tmp/pip-req-build-802rb0ax
  Running command git clone --filter=blob:none --quiet https://github.com/vsmalladi/FAIRBio /tmp/pip-req-build-802rb0ax
  Resolved https://github.com/vsmalladi/FAIRBio to commit 662095fef1cdd93d3446b14bb3c3e6df2e208915
  Preparing metadata (setup.py) ... done
  Created wheel for fairbio: filename=fairbio-0.1.0-py3-none-any.whl size=18426 sha256=70963c2654f1bd368028718ee61165ba339edfdc81d577a2b08e520a7ff55641
  Stored in directory: /tmp/pip-ephem-wheel-cache-ckc68a8k/wheels/f3/84/13/94ff52c8516b324e2df2c98669fde34a6fcab49ec51757c29a
Successfully built fairbio


## Download workflows from Dockstore


In [11]:
!fairbio-trs -v -r https://dockstore.org/api tools --all --toolclass Workflow -o dockstore_workflows.json

🔍 Fetching ALL tools from https://dockstore.org/api...
   (This may take a while, automatically paginating through all results...)
Starting new HTTPS connection (1): dockstore.org:443
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?limit=100&offset=0&toolClass=Workflow HTTP/1.1" 200 None
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?toolClass=Workflow&limit=100&offset=1 HTTP/1.1" 200 None
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?toolClass=Workflow&limit=100&offset=2 HTTP/1.1" 200 None
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?toolClass=Workflow&limit=100&offset=3 HTTP/1.1" 200 None
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?toolClass=Workflow&limit=100&offset=4 HTTP/1.1" 200 None
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?toolClass=Workflow&limit=100&offset=5 HTTP/1.1" 200 None
https://dockstore.org:443 "GET /api/ga4gh/trs/v2/tools?toolClass=Workflow&limit=100&offset=6 HTTP/1.1" 200 None
https://dockstore.org:443 "GET /

## Download workflows from Workflowhub

In [12]:
!fairbio-trs -v -r https://workflowhub.eu/ tools --all -o workflowhub.json

🔍 Fetching ALL tools from https://workflowhub.eu/...
   (This may take a while, automatically paginating through all results...)
Starting new HTTPS connection (1): workflowhub.eu:443
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=0 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=100 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=200 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=300 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=400 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=500 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=600 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=700 HTTP/1.1" 200 None
https://workflowhub.eu:443 "GET /ga4gh/trs/v2/tools?limit=100&offset=800 HTTP/1.1" 

## Unique Workflows


In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Debugging: Print first few lines of dockstore_workflows.json ---
try:
    with open('dockstore_workflows.json', 'r') as f:
        print("--- Start of dockstore_workflows.json content ---")
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 9: # Print first 10 lines
                break
        print("--- End of dockstore_workflows.json content ---")
except FileNotFoundError:
    print("Error: dockstore_workflows.json not found.")
except Exception as e:
    print(f"Error reading dockstore_workflows.json: {e}")
# -------------------------------------------------------------------

def process_workflow_data(filepath, source_name):
    """Loads and processes workflow data from a JSON file."""
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from {filepath}: {e}")
        return [] # Return empty list if JSON is malformed
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return []

    processed_entries = []
    unique_workflow_descriptors = set()

    # Ensure 'data' is iterable. If json.load returned a dict or other non-iterable,
    # converting to a list containing that item (if it makes sense) or handling as error.
    if not isinstance(data, list):
        # If it's a dict, perhaps it's a single workflow, put it in a list.
        if isinstance(data, dict):
            data = [data]
        else:
            print(f"Warning: Expected JSON data to be a list or dict, but got {type(data)} from {filepath}")
            return [] # Cannot process non-list/dict data


    for workflow_item in data:
        workflow = workflow_item
        # If 'workflow_item' is a string, it might be a stringified JSON object
        if isinstance(workflow_item, str):
            try:
                workflow = json.loads(workflow_item)
            except json.JSONDecodeError:
                print(f"Warning: Could not parse string element as JSON in {filepath}: {workflow_item[:100]}...")
                continue # Skip if parsing fails

        # Now, 'workflow' should be a dictionary. If not, skip.
        if not isinstance(workflow, dict):
            print(f"Warning: Expected workflow to be a dictionary, but got {type(workflow)} after processing in {filepath}. Item: {workflow_item}")
            continue

        workflow_id = workflow.get('id')
        if not workflow_id:
            # print(f"Warning: Workflow dictionary missing 'id' in {filepath}. Item: {workflow}")
            continue

        versions = workflow.get('versions', [])
        if not isinstance(versions, list):
            # print(f"Warning: 'versions' is not a list for workflow_id {workflow_id} in {filepath}. Type: {type(versions)}")
            versions = []

        for version in versions:
            if not isinstance(version, dict):
                # print(f"Warning: Expected version to be a dictionary, but got {type(version)} for workflow_id {workflow_id} in {filepath}. Item: {version}")
                continue

            descriptor_types = version.get('descriptor_type', []) # Get as list, default to empty list
            if isinstance(descriptor_types, list):
                for dt_item in descriptor_types:
                    if dt_item and (workflow_id, dt_item) not in unique_workflow_descriptors:
                        processed_entries.append({'Source': source_name, 'Workflow_ID': workflow_id, 'Descriptor_Type': dt_item})
                        unique_workflow_descriptors.add((workflow_id, dt_item))
            elif isinstance(descriptor_types, str): # Handle case where it might be a single string for some reason
                 if descriptor_types and (workflow_id, descriptor_types) not in unique_workflow_descriptors:
                    processed_entries.append({'Source': source_name, 'Workflow_ID': workflow_id, 'Descriptor_Type': descriptor_types})
                    unique_workflow_descriptors.add((workflow_id, descriptor_types))
    return processed_entries

# Process data from both files
dockstore_processed = process_workflow_data('dockstore_workflows.json', 'Dockstore')
workflowhub_processed = process_workflow_data('workflowhub.json', 'WorkflowHub')

# Combine and create DataFrame
all_workflow_data = dockstore_processed + workflowhub_processed
df = pd.DataFrame(all_workflow_data)

# Count unique Workflow_IDs per Source and Descriptor_Type
# Using nunique() ensures that if a single workflow has multiple versions of the same descriptor type,
# it's still counted only once for that descriptor type.
if not df.empty:
    unique_counts = df.groupby(['Source', 'Descriptor_Type'])['Workflow_ID'].nunique().reset_index()

    # Plotting
    plt.figure(figsize=(12, 7))
    sns.barplot(data=unique_counts, x='Descriptor_Type', y='Workflow_ID', hue='Source', palette='viridis')
    plt.title('Unique Workflow Tools by Descriptor Type from Dockstore and WorkflowHub')
    plt.xlabel('Workflow Descriptor Type')
    plt.ylabel('Number of Unique Workflow Tools')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()
else:
    print("No workflow data found to plot.")

Error: dockstore_workflows.json not found.
Error: File not found at dockstore_workflows.json
Error: File not found at workflowhub.json
No workflow data found to plot.
